In [21]:

from sklearn.datasets import fetch_20newsgroups
import pandas as pd
from nltk.tokenize import word_tokenize
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense

clean_params = {'remove': ('headers', 'footers', 'quotes')}

category_mapping = {
    'misc.forsale': 'Business',
    'rec.motorcycles': 'Entertainment',
    'talk.politics.misc': 'Politics',
    'rec.sport.baseball': 'Sport',
    'sci.electronics': 'Tech'
}

news_dataset = fetch_20newsgroups(subset='all', categories=list(category_mapping.keys()), **clean_params)
print(news_dataset.data)
print(news_dataset.target_names)

df = pd.DataFrame({"text": news_dataset.data, "target": news_dataset.target})


def check_for_empty_string(df, param):
    return df[df[param].str.strip().str.len() > 0].copy()


def check_for_empty_list(df, param):
    return df[df[param].str.len() > 0].copy()


def clean_and_tokenize(text):
    return [w for w in word_tokenize(text.lower()) if w.isalpha()]


df = check_for_empty_string(df, 'text')

df['tokens'] = df['text'].apply(clean_and_tokenize)
df = check_for_empty_list(df, 'tokens')
df['clean_str'] = df['tokens'].apply(lambda x: " ".join(x))
print(df.head())

# Train / Test Split (80/20)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    df, df['target'].values, test_size=0.2, random_state=42, stratify=df['target'].values
)

result = []


def run_logistic_regression(x_train, x_test, y_train, y_test, params):
    lr = LogisticRegression(max_iter=500, random_state=42)
    lr.fit(x_train, y_train)
    predict = lr.predict(x_test)
    return accuracy_score(y_test, predict)


def run_lstm(x_train, x_test, y_train, y_test, params, epochs=3):
    input_shape = (x_train.shape[1], x_train.shape[2])
    # Convert labels for LSTM
    num_classes = 5
    y_train_lstm = to_categorical(y_train, num_classes=num_classes)
    y_test_lstm = to_categorical(y_test, num_classes=num_classes)
    model = Sequential([
        Input(shape=input_shape),
        LSTM(256, return_sequences=True),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train_lstm, epochs=epochs, batch_size=64, verbose=0)
    _, acc = model.evaluate(x_test, y_test_lstm, verbose=0)
    return acc




IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



                                                text  target  \
0  Ladies and gentleman,\n\n\tStep one was taken ...       2   
1  \n\n\n\nIt has been a while since I knew the e...       3   
2  So THAT'S what happened to Denny McLain.  Sad....       2   
3  MLB Standings and Scores for Thursday, April 1...       2   
4  \n\n\tAnd the Commonwealth of Virginia has not...       3   

                                              tokens  \
0  [ladies, and, gentleman, step, one, was, taken...   
1  [it, has, been, a, while, since, i, knew, the,...   
2  [so, that, what, happened, to, denny, mclain, ...   
3  [mlb, standings, and, scores, for, thursday, a...   
4  [and, the, commonwealth, of, virginia, has, no...   

                                           clean_str  
0  ladies and gentleman step one was taken on the...  
1  it has been a while since i knew the electroni...  
2  so that what happened to denny mclain sad for ...  
3  mlb standings and scores for thursday april in...  
4  